<a href="https://colab.research.google.com/github/DanishShah619/LibraryIq/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
%pip install fastapi uvicorn[standard] pydantic pydantic-settings langchain-community langchain-chroma langchain-text-splitters sentence-transformers pandas numpy redis python-dotenv chromadb pyngrok

In [25]:
import os, shutil

SRC = "/content/drive/MyDrive/ml_book_recommendationmodel"

for fname in ["main.py", "recommender.py", "config.py"]:
    shutil.copy(f"{SRC}/{fname}", f"/content/{fname}")
    print(f"✅ {fname} copied")

os.chdir("/content")
print(f"\nAll files in /content/: {os.listdir('/content/')}")

✅ main.py copied
✅ recommender.py copied
✅ config.py copied

All files in /content/: ['.config', 'recommender.py', 'config.py', '__pycache__', 'drive', 'main.py', 'sample_data']


In [26]:
# Install and start Redis
import subprocess, time

subprocess.run(["apt-get", "install", "-y", "redis-server"], capture_output=True)
subprocess.Popen(["redis-server", "--daemonize", "yes"])

time.sleep(2)  # give it a moment to start

# Verify it's running
result = subprocess.run(["redis-cli", "ping"], capture_output=True, text=True)
print("Redis status:", result.stdout.strip())  # should print: PONG

Redis status: PONG


In [43]:
import subprocess, threading, time
from pyngrok import ngrok, conf

# Paste your ngrok token here
NGROK_TOKEN = "ngork_token"
conf.get_default().auth_token = NGROK_TOKEN

# Set secret — must match ML_SERVICE_SECRET in your Next.js .env
ML_SECRET = "ml_secret"
import os
os.environ["ML_SERVICE_SECRET"] = ML_SECRET

# Start FastAPI in a background thread
def run_server():
    os.chdir("/content")
    subprocess.run(["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"])

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

# Wait for server to come up (first start builds the ChromaDB index — ~60-90s)
print("Waiting for server to start and build index...")
time.sleep(200)

# Open ngrok tunnel
public_url = ngrok.connect(8000)
print(f"\n✅ ML Service is live at: {public_url}")
print(f"\nSet this in your Next.js .env:")
print(f"  ML_SERVICE_URL={public_url}")

Waiting for server to start and build index...

✅ ML Service is live at: NgrokTunnel: "https://audacity-dispersal-numeric.ngrok-free.dev" -> "http://localhost:8000"

Set this in your Next.js .env:
  ML_SERVICE_URL=NgrokTunnel: "https://audacity-dispersal-numeric.ngrok-free.dev" -> "http://localhost:8000"


In [39]:
import subprocess

# Kill any existing uvicorn on port 8000
subprocess.run(["fuser", "-k", "8000/tcp"], capture_output=True)
print("✅ Cleared port 8000")

# Fix Redis URL directly in the environment
import os
os.environ["REDIS_URL"] = "redis://localhost:6379"

# Verify
print("REDIS_URL:", os.environ.get("REDIS_URL"))

✅ Cleared port 8000
REDIS_URL: redis://localhost:6379


In [45]:
import requests

BASE = str("https://audacity-dispersal-numeric.ngrok-free.dev")  # the ngrok URL from Cell 4
HEADERS = {"Authorization": f"Bearer {ML_SECRET}"}

# Health check
r = requests.get(f"{BASE}/health")
print("Health:", r.json())

# Quick recommendation test
r = requests.post(f"{BASE}/predict", headers=HEADERS, json={
    "query": "a dark mystery with unexpected twists",
    "category": "All",
    "tone": "Suspenseful",
    "top_k": 5
})
data = r.json()
print(f"\nGot {data['count']} recommendations in {data['latency_ms']}ms")
for book in data["recommendations"]:
    print(f"  - {book['title']} (confidence: {book['confidence']})")

Health: {'status': 'ok', 'books_loaded': 5197, 'cache_available': True}

Got 5 recommendations in 0.46ms
  - Switch on the Night (confidence: 0.1542)
  - Mystery Play (confidence: 0.1946)
  - The Girl Who Loved Tom Gordon (confidence: 0.0246)
  - Strangers (confidence: 0.1249)
  - The Taking (confidence: 0.1391)
